In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# BASICS

In [3]:
df = pd.read_csv("ObesityDataSet2.csv", sep = ',')

In [4]:
df

,Gender,Age,Height,Weight,FamilyHistory,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,Result
0,Female,24,1.58,65.29,yes,no,2.03,2.74,Sometimes,no,2.00,no,1.28,1.019,no,Public_Transportation,Overweight_Level_II
1,Male,23,1.65,66.00,no,no,3.00,3.00,Sometimes,no,2.00,no,3.00,0.000,no,Public_Transportation,Normal_Weight
2,Female,21,1.69,51.26,yes,yes,3.00,3.18,Frequently,no,1.91,no,0.48,0.625,no,Public_Transportation,Insufficient_Weight
3,Female,22,1.69,65.00,yes,yes,2.00,3.00,Sometimes,no,2.00,no,1.00,1.000,Sometimes,Public_Transportation,Normal_Weight
4,Female,23,1.61,82.64,yes,yes,2.96,1.00,Sometimes,no,2.98,no,0.74,2.000,Sometimes,Public_Transportation,Obesity_Type_I
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1051,Female,21,1.74,130.93,yes,yes,3.00,3.00,Sometimes,no,1.85,no,1.46,0.962,Sometimes,Public_Transportation,Obesity_Type_III
1052,Female,17,1.54,57.26,no,yes,1.97,2.34,Sometimes,no,1.71,yes,0.10,1.191,Sometimes,Public_Transportation,Overweight_Level_I
1053,Female,19,1.60,45.00,no,no,3.00,3.00,no,no,3.00,yes,2.00,0.000,no,Walking,Insufficient_Weight
1054,Female,25,1.69,113.45,yes,yes,3.00,3.00,Sometimes,no,2.99,no,0.39,0.284,Sometimes,Public_Transportation,Obesity_Type_III


In [5]:
# Rename family history and output for convenience (changed in the csv)

# Nulls and Duplicates

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1056 entries, 0 to 1055
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Gender         1056 non-null   object 
 1   Age            1056 non-null   object 
 2   Height         1056 non-null   float64
 3   Weight         1056 non-null   float64
 4   FamilyHistory  1056 non-null   object 
 5   FAVC           1056 non-null   object 
 6   FCVC           1019 non-null   float64
 7   NCP            1056 non-null   float64
 8   CAEC           1056 non-null   object 
 9   SMOKE          1056 non-null   object 
 10  CH2O           1056 non-null   float64
 11  SCC            1056 non-null   object 
 12  FAF            1056 non-null   float64
 13  TUE            1056 non-null   float64
 14  CALC           1056 non-null   object 
 15  MTRANS         1030 non-null   object 
 16  Result         1056 non-null   object 
dtypes: float64(7), object(10)
memory usage: 140.4+ KB


In [8]:
df.duplicated().sum()

10

We have a lot of data, we can drop NA and duplicates to prioritze data integrity

In [10]:
df = df.dropna()
df = df.drop_duplicates()

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 984 entries, 0 to 1055
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Gender         984 non-null    object 
 1   Age            984 non-null    object 
 2   Height         984 non-null    float64
 3   Weight         984 non-null    float64
 4   FamilyHistory  984 non-null    object 
 5   FAVC           984 non-null    object 
 6   FCVC           984 non-null    float64
 7   NCP            984 non-null    float64
 8   CAEC           984 non-null    object 
 9   SMOKE          984 non-null    object 
 10  CH2O           984 non-null    float64
 11  SCC            984 non-null    object 
 12  FAF            984 non-null    float64
 13  TUE            984 non-null    float64
 14  CALC           984 non-null    object 
 15  MTRANS         984 non-null    object 
 16  Result         984 non-null    object 
dtypes: float64(7), object(10)
memory usage: 138.4+ KB


# Feature Engineering

## Age

In [14]:
df['Age'] = df['Age'].astype(str).str.replace(' years', '', regex=False).astype(int)

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 984 entries, 0 to 1055
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Gender         984 non-null    object 
 1   Age            984 non-null    int32  
 2   Height         984 non-null    float64
 3   Weight         984 non-null    float64
 4   FamilyHistory  984 non-null    object 
 5   FAVC           984 non-null    object 
 6   FCVC           984 non-null    float64
 7   NCP            984 non-null    float64
 8   CAEC           984 non-null    object 
 9   SMOKE          984 non-null    object 
 10  CH2O           984 non-null    float64
 11  SCC            984 non-null    object 
 12  FAF            984 non-null    float64
 13  TUE            984 non-null    float64
 14  CALC           984 non-null    object 
 15  MTRANS         984 non-null    object 
 16  Result         984 non-null    object 
dtypes: float64(7), int32(1), object(9)
memory usage: 134.5+ KB

## Check Numerical Distribution

In [17]:
df.describe()

,Age,Height,Weight,FCVC,NCP,CH2O,FAF,TUE
count,984.000000,984.000000,984.000000,984.000000,984.000000,984.000000,984.000000,984.000000
mean,24.251016,1.702002,86.909228,2.425671,2.663181,1.996250,1.006951,0.650186
std,6.250697,0.091622,26.368818,0.535816,0.796000,0.606549,0.850724,0.616592
min,15.000000,1.450000,39.000000,1.000000,1.000000,1.000000,0.000000,0.000000
25%,20.000000,1.630000,66.000000,2.000000,2.587500,1.530000,0.130000,0.000000
50%,23.000000,1.700000,83.000000,2.405000,3.000000,2.000000,1.000000,0.607500
75%,26.000000,1.770000,108.255000,3.000000,3.000000,2.440000,1.640000,1.000000
max,55.000000,1.950000,173.000000,3.000000,4.000000,3.000000,3.000000,2.000000


All variables are already within specified ranges, especially for FCVC, CH20, FAF, TUE

## Check Categoricals

In [20]:
for col in df.select_dtypes(include=['object', 'category','float64']).columns:
    print(df[col].value_counts())
    print()

Gender
Male      497
Female    487
Name: count, dtype: int64

Height
1.75    60
1.70    58
1.76    50
1.65    48
1.63    40
1.60    37
1.62    36
1.71    35
1.77    34
1.72    34
1.69    33
1.61    32
1.80    32
1.64    30
1.66    30
1.67    30
1.78    26
1.68    25
1.73    25
1.82    25
1.85    23
1.74    23
1.79    22
1.56    17
1.84    16
1.81    15
1.83    15
1.55    14
1.58    13
1.59    13
1.54    12
1.53    11
1.50    10
1.57    10
1.87     8
1.91     7
1.86     6
1.52     6
1.89     5
1.88     4
1.90     4
1.92     3
1.51     3
1.95     1
1.93     1
1.45     1
1.49     1
Name: count, dtype: int64

Weight
80.00     34
75.00     18
50.00     17
60.00     15
42.00     14
          ..
173.00     1
93.09      1
133.68     1
84.56      1
118.28     1
Name: count, Length: 681, dtype: int64

FamilyHistory
yes    811
no     173
Name: count, dtype: int64

FAVC
yes    870
no     114
Name: count, dtype: int64

FCVC
3.00    313
2.00    283
1.00     17
2.88      9
2.05      9
       ... 
1.6

Many variables are yes/no  
MTRANS will use One-Hot  
Gender will also be labeled for simplicity  
CAEC, CALC, Result will use ordinal

# Encoding

In [23]:
yn_lab = ['no','yes']
mf_lab = ['Male', 'Female']
fq_ord = ['no', 'Sometimes', 'Frequently', 'Always']
rs_ord = ['Insufficient_Weight', 'Normal_Weight',
           'Overweight_Level_I', 'Overweight_Level_II',
           'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III']
lab_col = ['FamilyHistory', 'FAVC', 'SMOKE', 'SCC']

In [24]:
# RESULT
out_enc = OrdinalEncoder(categories = [rs_ord])
df['Result'] = out_enc.fit_transform(df[['Result']].values.reshape(-1, 1))

# CAEC & CALC
caec_enc = OrdinalEncoder(categories = [fq_ord])
df['CAEC'] = caec_enc.fit_transform(df[['CAEC']].values.reshape(-1, 1))

calc_enc = OrdinalEncoder(categories = [fq_ord])
df['CALC'] = calc_enc.fit_transform(df[['CALC']].values.reshape(-1, 1))

# Gender
gen_enc = OrdinalEncoder(categories = [mf_lab])
df['Gender'] = gen_enc.fit_transform(df[['Gender']].values.reshape(-1, 1))

# Others:
fam_hist_enc = OrdinalEncoder(categories = [yn_lab])
df['FamilyHistory'] = fam_hist_enc.fit_transform(df[['FamilyHistory']].values.reshape(-1, 1))

favc_enc = OrdinalEncoder(categories = [yn_lab])
df['FAVC'] = favc_enc.fit_transform(df[['FAVC']].values.reshape(-1, 1))

smoke_enc = OrdinalEncoder(categories = [yn_lab])
df['SMOKE'] = smoke_enc.fit_transform(df[['SMOKE']].values.reshape(-1, 1))

scc_enc = OrdinalEncoder(categories = [yn_lab])
df['SCC'] = scc_enc.fit_transform(df[['SCC']].values.reshape(-1, 1))

In [25]:
ohe_enc = OneHotEncoder(sparse_output=False, drop='first')

enc_arr = ohe_enc.fit_transform(df[['MTRANS']])
enc_df = pd.DataFrame(enc_arr, columns=ohe_enc.get_feature_names_out(['MTRANS']))

df.drop(columns=['MTRANS'], inplace=True)
df = pd.concat([df.reset_index(drop=True), enc_df.reset_index(drop=True)], axis=1)

In [26]:
df.head()

,Gender,Age,Height,Weight,FamilyHistory,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,Result,MTRANS_Bike,MTRANS_Motorbike,MTRANS_Public_Transportation,MTRANS_Walking
0,1.0,24,1.58,65.29,1.0,0.0,2.03,2.74,1.0,0.0,2.00,0.0,1.28,1.019,0.0,3.0,0.0,0.0,1.0,0.0
1,0.0,23,1.65,66.00,0.0,0.0,3.00,3.00,1.0,0.0,2.00,0.0,3.00,0.000,0.0,1.0,0.0,0.0,1.0,0.0
2,1.0,21,1.69,51.26,1.0,1.0,3.00,3.18,2.0,0.0,1.91,0.0,0.48,0.625,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,22,1.69,65.00,1.0,1.0,2.00,3.00,1.0,0.0,2.00,0.0,1.00,1.000,1.0,1.0,0.0,0.0,1.0,0.0
4,1.0,23,1.61,82.64,1.0,1.0,2.96,1.00,1.0,0.0,2.98,0.0,0.74,2.000,1.0,4.0,0.0,0.0,1.0,0.0


# Training

## Train-Test Split

split 80-20, train-test

In [30]:
x = df.drop(columns=['Result'])
y = df['Result']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=354)

## Pre-GridSearch

In [32]:
# Random Forest
rf_model = RandomForestClassifier(random_state=354)
rf_model.fit(x_train, y_train)
rf_pred = rf_model.predict(x_test)

# XG Boost
xg_model = xgb.XGBClassifier(random_state=354,
                             tree_method='gpu_hist',
                             predictor='gpu_predictor',)
xg_model.fit(x_train, y_train)
xg_pred = xg_model.predict(x_test)

C:\Users\kynes\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [06:01:04] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
C:\Users\kynes\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [06:01:04] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\kynes\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [06:01:05] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgbo

In [33]:
print("\n\n__________RANDOM FOREST__________")
print("Classification Report:\n", classification_report(y_test, rf_pred, zero_division=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_pred))

print("\n\n__________XG BOOST__________")
print("Classification Report:\n", classification_report(y_test, xg_pred, zero_division=1))
print("Confusion Matrix:\n", confusion_matrix(y_test, xg_pred))



__________RANDOM FOREST__________
Classification Report:
               precision    recall  f1-score   support

         0.0       1.00      0.95      0.98        22
         1.0       0.76      1.00      0.86        22
         2.0       1.00      0.86      0.93        29
         3.0       0.97      0.93      0.95        30
         4.0       0.94      0.97      0.96        33
         5.0       1.00      0.97      0.99        34
         6.0       1.00      0.96      0.98        27

    accuracy                           0.95       197
   macro avg       0.95      0.95      0.95       197
weighted avg       0.96      0.95      0.95       197

Confusion Matrix:
 [[21  1  0  0  0  0  0]
 [ 0 22  0  0  0  0  0]
 [ 0  3 25  1  0  0  0]
 [ 0  1  0 28  1  0  0]
 [ 0  1  0  0 32  0  0]
 [ 0  0  0  0  1 33  0]
 [ 0  1  0  0  0  0 26]]


__________XG BOOST__________
Classification Report:
               precision    recall  f1-score   support

         0.0       1.00      1.00      1.00  

Both models have great initial accuracies, with great predictions as shown in the confusion matrix

## Grid Search

don't use too many params, big dataset will take super long

In [37]:
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20],
    'min_samples_split': [2],
    'min_samples_leaf': [1],
    'max_features': ['sqrt'],
    'bootstrap': [True],
    'criterion': ['gini']
}

xg_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [4, 6],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'gamma': [0],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [1]
}

In [38]:
# Random Forest
rf_gs = GridSearchCV(estimator = rf_model,
                     param_grid = rf_param_grid,
                     cv=3, n_jobs=-1, scoring = 'accuracy')
rf_gs.fit(x_train, y_train)

# XG Boost
xg_gs = GridSearchCV(estimator = xg_model,
                     param_grid = xg_param_grid, 
                     cv=3, n_jobs=-1, scoring = 'accuracy')
xg_gs.fit(x_train, y_train)

C:\Users\kynes\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [06:02:56] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
C:\Users\kynes\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [06:02:56] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


GridSearchCV(cv=3,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, gamma=None,
                                     grow_policy=None, importance_type=None,
                                     interaction_constraints=None,
                                     learning_rate=None,...
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None,
                                     objective='multi:softprob', ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.8], 'gamma': [0],
                         'learning_rate': [0.05, 0.1], 'max_depth': [4, 6],
                         'n_estimators': [100, 200], 'reg_alpha': [0, 0.1],
                         'reg_lambda': [1], 'subsample': [0.8]},
             scoring='accuracy')

In [39]:
print("\n__________RANDOM FOREST__________")
print("Best score:", rf_gs.best_score_)
print("Best parameters:", rf_gs.best_params_)

print("\n__________XG BOOST__________")
print("Best score:", xg_gs.best_score_)
print("Best parameters:", xg_gs.best_params_)


__________RANDOM FOREST__________
Best score: 0.9186718137752882
Best parameters: {'bootstrap': True, 'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}

__________XG BOOST__________
Best score: 0.9275534786520767
Best parameters: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 200, 'reg_alpha': 0.1, 'reg_lambda': 1, 'subsample': 0.8}


## Deciding The Model

Random Forest did better than XGBoost in both, base and tuned situations

In [68]:
rf_best = rf_gs.best_estimator_
rf_grid = rf_best.predict(x_test)
xg_best = xg_gs.best_estimator_
xg_grid = xg_best.predict(x_test)

C:\Users\kynes\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [06:05:35] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


In [72]:
accuracy_score(y_test, rf_pred)

print(f"RF Base: {accuracy_score(y_test, rf_pred):.8f}")
print(f"RF Grid: {accuracy_score(y_test, rf_grid):.8f}")
print(f"XGB Base: {accuracy_score(y_test, xg_pred):.8f}")
print(f"XGB Grid: {accuracy_score(y_test, xg_grid):.8f}")

RF Base: 0.94923858
RF Grid: 0.92893401
XGB Base: 0.94923858
XGB Grid: 0.93908629


Hence, take the RF/XGB base model

## Save with pickle

In [46]:
import pickle

with open('rf_md_fp.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

In [164]:
## Save Encoders
ordinal_encoders_map = {
    'Gender': gen_enc,
    'FamilyHistory': fam_hist_enc,
    'FAVC': favc_enc,
    'SMOKE': smoke_enc,
    'SCC': scc_enc,
    'CAEC': caec_enc,
    'CALC': calc_enc
}

with open('ordinal_encoders.pkl', 'wb') as f:
    pickle.dump(ordinal_encoders_map, f)

with open('onehot_encoder.pkl', 'wb') as f:
    pickle.dump(ohe_enc, f)

with open('target_encoder.pkl', 'wb') as f:
    pickle.dump(out_enc, f)

with open('model_features_order.pkl', 'wb') as f:
    pickle.dump(x.columns.tolist(), f)